In [ ]:
!pip uninstall -y \
langchain \
langchain-core \
langchain-community \
langchain-google-genai \
langchain-huggingface \
langchain-text-splitters

In [ ]:
!pip install -q \
protobuf==5.29.6 \
tensorflow==2.20.0 \
transformers==4.53.3 \
sentence-transformers==5.1.0

In [ ]:
!pip install -U \
langchain==1.0.0 \
langchain-core==1.0.0 \
langchain-community \
langchain-google-genai \
langchain-huggingface \
langchain-text-splitters \
faiss-cpu \
pypdf \
sentence-transformers

  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_google_genai-4.2.5-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.4-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.3-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadat

In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnablePassthrough

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Sentence Transformers Loaded!")

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print(llm.invoke("Hello").content)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Hello! Tell me who you are.")
print(response.content)

In [ ]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

print("All imports successful!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Total Pages:", len(documents))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings Loaded Successfully ✅")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embedding
)

print("FAISS database created successfully!")

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain Created Successfully!")

In [ ]:
while True:
    question = input("Ask Question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    answer = rag_chain.invoke(question)

    print("\n" + "="*60)
    print("Answer:\n")
    print(answer)
    print("="*60)